In [26]:
import sqlite3
import requests
from bs4 import BeautifulSoup
import pandas as pd



# 2024 WWW Accepted Papers

In [40]:

def create_database(DB_PATH):
    """Create the database and the Conference table."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Conference (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        Title TEXT NOT NULL,
        Author TEXT NOT NULL,
        PDF_Link TEXT,
        Code_URL TEXT,
        Conference_Name TEXT NOT NULL
    )
    ''')

    print("Conference 테이블이 생성되었습니다.")
    conn.commit()
    conn.close()

def save_to_database(df, conference_name, DB_PATH):
    conn = sqlite3.connect(DB_PATH, timeout=10)
    cursor = conn.cursor()

    try:
        for _, row in df.iterrows():  # ✅ iterrows() 사용하여 DataFrame의 각 행을 처리
            # 중복 데이터 확인
            cursor.execute('''
            SELECT 1 FROM Conference WHERE Title = ? AND Author = ? AND Conference_Name = ?
            ''', (row['title'], row['authors'], conference_name))
            result = cursor.fetchone()

            if not result:
                cursor.execute('''
                INSERT INTO Conference (Title, Author, PDF_Link, Code_URL, Conference_Name)
                VALUES (?, ?, ?, ?, ?)
                ''', (row['title'], row['authors'], row['pdf_link'], row['code_url'], conference_name))

        conn.commit()
        print(f"{len(df)}개의 논문이 {conference_name}에 저장되었습니다.")
    except sqlite3.Error as e:
        print(f"Database error: {e}")
    finally:
        conn.close()


def get_www_papers(page_link, conference_name):
    """WWW 2024 학회의 Accepted Papers 페이지에서 논문 정보 크롤링"""
    page_content = requests.get(page_link)
    
    if page_content is None:
        return []

    soup = BeautifulSoup(page_content.text, "html.parser")

    # HTML을 저장 (디버깅용)
    with open("www2024_accepted_papers.html", "w", encoding="utf-8") as f:
        f.write(soup.prettify())

    papers = []

    # 논문 리스트가 포함된 div 태그 찾기
    for card in soup.find_all("div", class_="card my-2 p-2"):
        title_tag = card.find("strong")
        author_tag = card.find("p", class_="m-0 p-0")
        pdf_link_tag = card.find("a", class_="btn btn-sm bg-primary my-1")
        code_url_tag = card.find("a", text="GitHub")  # GitHub 링크가 있을 경우

        title = title_tag.text.strip() if title_tag else None
        authors = author_tag.text.strip() if author_tag else None
        pdf_link = pdf_link_tag["href"] if pdf_link_tag else None
        code_url = code_url_tag["href"] if code_url_tag else None
        
        papers.append({"title": title, "authors": authors, "pdf_link": None, "code_url": pdf_link, 'conference_name': conference_name})

    return papers

In [30]:
page_link='https://www2024.thewebconf.org/accepted/research-tracks/' 
DB_PATH = "www_conference_2024.db"
conference_name = 'WWW 2024'
papers = get_www_papers(page_link, conference_name)

/tmp/ipykernel_2099291/2277661230.py:66: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  code_url_tag = card.find("a", text="GitHub")  # GitHub 링크가 있을 경우


In [31]:
response = requests.get(page_link) 
if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser') 
    with open('www2024_accepted_papers.html', 'w', encoding='utf-8') as f:
        f.write(soup.prettify()) 


In [34]:
import pandas as pd 


df_papers = pd.DataFrame(papers) 
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Intelligent Model Update Strategy for Sequenti...,"Zheqi Lv, Wenqiao Zhang, Zhengyu Chen, Shengyu...",None,None,WWW 2024
1,Advancing Web 3.0: Making Smart Contracts Smar...,"Junqin Huang, Linghe Kong, Guanjie Cheng, Qiao...",None,https://doi.org/10.5281/zenodo.10566861,WWW 2024
2,Efficient Computation of Signature-Restricted ...,Yizheng Zhao,None,None,WWW 2024
3,From Promises to Practice: Evaluating the Priv...,"Xiaoyin Liu, Wenzhi Li, Qinsheng Hou, Shishuai...",None,https://doi.org/10.5281/zenodo.10577793,WWW 2024
4,"LFDe: A Lighter, Faster and More Data-Efficien...","Zhigang Kan, Liwen Peng, Yifu Gao, Ning Liu, L...",None,None,WWW 2024


In [37]:
# DB 생성 
DB_PATH = "www_conference_2024.db" 
create_database(DB_PATH) 

Conference 테이블이 생성되었습니다.


In [41]:
save_to_database(df_papers, conference_name, DB_PATH)

405개의 논문이 WWW 2024에 저장되었습니다.


# 2023 WWW Accepted Papers

In [60]:
url="https://www.sigweb.org/toc/www23.html"
DB_PATH = 'www_conference_2023.db' 
create_database(DB_PATH) 

Conference 테이블이 생성되었습니다.


In [61]:
def get_www_papers(page_link, conference_name):
    """WWW 2023 학회의 Accepted Papers 페이지에서 논문 정보 크롤링"""
    
    # 웹페이지 요청
    response = requests.get(page_link)
    
    if response.status_code != 200:
        print("❌ 페이지를 불러오지 못했습니다.")
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    # HTML을 저장 (디버깅용)
    with open("www2023_accepted_papers.html", "w", encoding="utf-8") as f:
        f.write(soup.prettify())

    papers = []

    # 논문 제목과 저자 정보 추출
    for paper_section in soup.find_all("h3"):  # 논문 제목이 포함된 태그 찾기
        title_tag = paper_section.find("a")  # 제목이 포함된 a 태그 찾기
        author_list = paper_section.find_next_sibling("ul")  # 저자 목록이 포함된 ul 태그 찾기

        if title_tag and author_list:
            title_text = title_tag.text.strip()  # 제목 텍스트 추출
            authors = [li.text.strip() for li in author_list.find_all("li")]  # 저자 목록 추출
            author_text = ", ".join(authors)  # 저자 이름을 쉼표로 구분

            papers.append({
                "title": title_text,
                "authors": author_text,
                "pdf_link": None,
                "code_url": None,
                "conference_name": conference_name
            })

    return papers

In [57]:
papers = get_www_papers(url, 'WWW 2023')


[{'title': 'Using diversity as a source of scientific innovation for the Web', 'authors': 'Barbara Poblete', 'pdf_link': None, 'code_url': None, 'conference_name': 'WWW 2023'}, {'title': 'Decolonizing Creative Labor in the age of AI', 'authors': 'Payal Arora', 'pdf_link': None, 'code_url': None, 'conference_name': 'WWW 2023'}, {'title': 'Concept Regulation in the Social Sciences', 'authors': 'Zachary Elkins', 'pdf_link': None, 'code_url': None, 'conference_name': 'WWW 2023'}, {'title': 'GNNs and Graph Generative models for biomedical applications', 'authors': 'Michalis Vazirgiannis', 'pdf_link': None, 'code_url': None, 'conference_name': 'WWW 2023'}, {'title': 'CONNECTIVITY', 'authors': 'Robert Melancton Metcalfe', 'pdf_link': None, 'code_url': None, 'conference_name': 'WWW 2023'}, {'title': 'GELTOR: A Graph Embedding Method based on Listwise Learning to Rank', 'authors': 'Masoud Reyhani Hamedani, Jin-Su Ryu, Sang-Wook Kim', 'pdf_link': None, 'code_url': None, 'conference_name': 'WWW 2

In [58]:
df_papers = pd.DataFrame(papers) 
df_papers.head(30)

,title,authors,pdf_link,code_url,conference_name
0,Using diversity as a source of scientific inno...,Barbara Poblete,None,None,WWW 2023
1,Decolonizing Creative Labor in the age of AI,Payal Arora,None,None,WWW 2023
2,Concept Regulation in the Social Sciences,Zachary Elkins,None,None,WWW 2023
3,GNNs and Graph Generative models for biomedica...,Michalis Vazirgiannis,None,None,WWW 2023
4,CONNECTIVITY,Robert Melancton Metcalfe,None,None,WWW 2023
5,GELTOR: A Graph Embedding Method based on List...,"Masoud Reyhani Hamedani, Jin-Su Ryu, Sang-Wook...",None,None,WWW 2023
6,Graph-less Collaborative Filtering,"Lianghao Xia, Chao Huang, Jiao Shi, Yong Xu",None,None,WWW 2023
7,Fair Graph Representation Learning via Diverse...,"Zheyuan Liu, Chunhui Zhang, Yijun Tian, Erchi ...",None,None,WWW 2023
8,Multi-Aspect Heterogeneous Graph Augmentation,"Yuchen Zhou, Yanan Cao, Yongchao Liu, Yanmin S...",None,None,WWW 2023
9,Testing Cluster Properties of Signed Graphs,"Florian Adriaens, Simon Apers",None,None,WWW 2023


In [62]:
save_to_database(df_papers, conference_name= 'WWW 2023',DB_PATH=DB_PATH)

407개의 논문이 WWW 2023에 저장되었습니다.


# 2022 Accepted Papers

In [63]:
url="https://www.sigweb.org/toc/www22.html"
DB_PATH = 'www_conference_2022.db' 
create_database(DB_PATH) 

Conference 테이블이 생성되었습니다.


In [64]:
def get_www_papers(page_link, conference_name):
    """WWW 2022 학회의 Accepted Papers 페이지에서 논문 정보 크롤링"""
    
    # 웹페이지 요청
    response = requests.get(page_link)
    
    if response.status_code != 200:
        print("❌ 페이지를 불러오지 못했습니다.")
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    # HTML을 저장 (디버깅용)
    with open("www2022_accepted_papers.html", "w", encoding="utf-8") as f:
        f.write(soup.prettify())

    papers = []

    # 논문 제목과 저자 정보 추출
    for paper_section in soup.find_all("h3"):  # 논문 제목이 포함된 태그 찾기
        title_tag = paper_section.find("a")  # 제목이 포함된 a 태그 찾기
        author_list = paper_section.find_next_sibling("ul")  # 저자 목록이 포함된 ul 태그 찾기

        if title_tag and author_list:
            title_text = title_tag.text.strip()  # 제목 텍스트 추출
            authors = [li.text.strip() for li in author_list.find_all("li")]  # 저자 목록 추출
            author_text = ", ".join(authors)  # 저자 이름을 쉼표로 구분

            papers.append({
                "title": title_text,
                "authors": author_text,
                "pdf_link": None,
                "code_url": None,
                "conference_name": conference_name
            })

    return papers

In [65]:
papers = get_www_papers(url, 'WWW 2022')

In [66]:
df_papers = pd.DataFrame(papers) 
df_papers.head(5)

,title,authors,pdf_link,code_url,conference_name
0,WISE: Wavelet based Interpretable Stock Embedd...,"Mengying Zhu, Yan Wang, Fei Wu, Mengyuan Yang,...",None,None,WWW 2022
1,Cyclic Arbitrage in Decentralized Exchanges,"Ye Wang, Yan Chen, Haotian Wu, Liyi Zhou, Shui...",None,None,WWW 2022
2,An Exploratory Study of Stock Price Movements ...,"Sourav Medya, Mohammad Rasoolinejad, Yang Yang...",None,None,WWW 2022
3,DC-GNN: Decoupled Graph Neural Networks for Im...,"Chenchen Feng, Yu He, Shiyang Wen, Guojun Liu,...",None,None,WWW 2022
4,Multilingual Semantic Sourcing using Product I...,"Sourab Mangrulkar, Ankith M S, Vivek Sembium",None,None,WWW 2022


In [67]:
save_to_database(df_papers, conference_name= 'WWW 2022',DB_PATH=DB_PATH)

225개의 논문이 WWW 2022에 저장되었습니다.


# 2021 Accepted Papers

In [71]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

def get_www_papers(page_link, conference_name):
    """WWW 2021 학회의 Accepted Papers 페이지에서 논문 정보 크롤링 후 DataFrame 반환"""
    
    # 웹페이지 요청
    response = requests.get(page_link)
    
    if response.status_code != 200:
        print("❌ 페이지를 불러오지 못했습니다.")
        return pd.DataFrame()  # 빈 DataFrame 반환

    soup = BeautifulSoup(response.text, "html.parser")

    # HTML 저장 (디버깅용)
    with open("www2021_accepted_papers.html", "w", encoding="utf-8") as f:
        f.write(soup.prettify())

    papers = []

    # 논문 제목과 저자 정보 추출
    for row in soup.find_all("tr"):  # 각 논문이 포함된 행
        cells = row.find_all("td")

        if len(cells) >= 2:
            title_tag = cells[0].find("strong", style=lambda x: x and "color: #359c8d" in x)  # 논문 제목
            author_tag = cells[1]  # 저자 정보

            if title_tag and author_tag:
                title_text = title_tag.text.strip()

                # 저자 정보에서 "Authors:" 부분 제거 및 괄호 안의 대학/기관 정보 제거
                authors_raw = author_tag.text.replace("Authors:", "").strip()
                authors_cleaned = re.sub(r"\(.*?\)", "", authors_raw).strip()

                papers.append({
                    "title": title_text,
                    "authors": authors_cleaned,
                    "pdf_link": None,
                    "code_url": None,
                    "conference_name": conference_name
                })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers


In [72]:
# 웹페이지 URL 및 데이터베이스 설정
url = "https://archives.iw3c2.org/www2021/program/papers/"
DB_PATH = "www_conference_2021.db"

# 데이터베이스 생성
create_database(DB_PATH)

# 논문 데이터 크롤링
df_papers = get_www_papers(url, "WWW 2021")

# 크롤링된 논문 데이터 확인
df_papers.head()


Conference 테이블이 생성되었습니다.


,title,authors,pdf_link,code_url,conference_name
0,“Is it a Qoincidence?”: An Exploratory Study o...,"Antonis Papasavva , Jeremy Blackburn , Gianluc...",None,None,WWW 2021
1,“Short is the Road that Leads from Fear to Hat...,"Punyajoy Saha , Binny Mathew , Pawan Goyal , K...",None,None,WWW 2021
2,"âGo eat a bat, Chang!â: On the Emergence o...","Fatemeh Tahmasbi , Leonard Schild , Chen Ling ...",None,None,WWW 2021
3,#Twiti: Social Listening for Threat Intelligence,"Hyejin Shin , Woochul Shim , Saebom Kim , Sol ...",None,None,WWW 2021
4,A Cooperative Memory Network for Personalized ...,"Jiahuan Pei , Pengjie Ren and Maarten de Rijke .",None,None,WWW 2021


In [73]:
save_to_database(df_papers, conference_name= 'WWW 2021',DB_PATH=DB_PATH)

355개의 논문이 WWW 2021에 저장되었습니다.


# 2020 Accepted Papers

In [87]:
import pandas as pd
import re
from bs4 import BeautifulSoup

def get_www_papers(html_file_path, conference_name):
    """저장된 HTML 파일에서 WWW 학회의 Accepted Papers 정보를 크롤링 후 DataFrame 반환"""

    # HTML 파일 읽기
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []

    # 논문 제목, 저자, PDF 링크 추출
    for article in soup.find_all("div", class_="issue-item__content"):
        title_tag = article.find("h5", class_="issue-item__title")
        authors_tag = article.find("ul", {"aria-label": "authors"})
        pdf_link_tag = article.find("a", {"aria-label": "PDF"})  # PDF 링크 찾기

        if title_tag and authors_tag:
            title_text = title_tag.text.strip()

            # 저자 정보에서 괄호 속 대학/기관 제거
            authors_raw = ", ".join([a.text.strip() for a in authors_tag.find_all("span")])
            authors_cleaned = re.sub(r"\(.*?\)", "", authors_raw).strip()

            # PDF 링크 정리
            pdf_link = pdf_link_tag["href"] if pdf_link_tag else None

            papers.append({
                "title": title_text,
                "authors": authors_cleaned,
                "pdf_link": pdf_link,
                "code_url": None,
                "conference_name": conference_name
            })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers


In [82]:
url = "https://dl.acm.org/doi/proceedings/10.1145/3366423#heading1"
DB_PATH = "www_conference_2020.db"

In [83]:
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [88]:
df_papers = get_www_papers("/home/cvlab/papers-update/conference/html/www2020_accepted_papers.html", "WWW 2020")

In [89]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Relation Adversarial Network for Low Resource ...,"Ningyu Zhang, ,, Shumin Deng, ,, Zhanlin Sun, ...",https://dl.acm.org/doi/pdf/10.1145/3366423.338...,None,WWW 2020
1,Learning to Classify: A Flow-Based Relation Ne...,"Wenbo Zheng, ,, Chao Gou, ,, Lan Yan, ,, Shaoc...",https://dl.acm.org/doi/pdf/10.1145/3366423.338...,None,WWW 2020
2,FiDo: Ubiquitous Fine-Grained WiFi-based Local...,"Xi Chen, ,, Hang Li, ,, Chenyi Zhou, ,, Xue Li...",https://dl.acm.org/doi/pdf/10.1145/3366423.338...,None,WWW 2020
3,An Empirical Study of the Use of Integrity Ver...,"Bertil Chapuis, ,, Olamide Omolola, ,, Mauro C...",https://dl.acm.org/doi/pdf/10.1145/3366423.338...,None,WWW 2020
4,Power-Law Graphs Have Minimal Scaling of Kemen...,"Wanyue Xu, ,, Yibin Sheng, ,, Zuobai Zhang, ,,...",https://dl.acm.org/doi/pdf/10.1145/3366423.338...,None,WWW 2020


In [90]:
save_to_database(df_papers, conference_name= 'WWW 2020',DB_PATH=DB_PATH)

219개의 논문이 WWW 2020에 저장되었습니다.


# 2019 Accepted Papers

In [115]:
import pandas as pd
import re
import requests
from bs4 import BeautifulSoup

def get_www_papers(page_link, conference_name):
    """WWW 학회의 Accepted Papers 페이지에서 논문 정보 크롤링 후 DataFrame 반환"""

    # 웹페이지 요청
    response = requests.get(page_link)
    
    if response.status_code != 200:
        print("❌ 페이지를 불러오지 못했습니다.")
        return pd.DataFrame()  # 빈 DataFrame 반환

    soup = BeautifulSoup(response.text, "html.parser")

    # HTML 저장 (디버깅용)
    with open("www2018_accepted_papers.html", "w", encoding="utf-8") as f:
        f.write(soup.prettify())

    papers = []

    # 논문 제목과 저자, PDF 링크, Code URL 추출
    for article in soup.find_all("div", class_="issue-item-container"):
        title_tag = article.find("h5", class_="issue-item__title")
        authors_tag = article.find("ul", {"aria-label": "authors"})
        pdf_link_tag = article.find("a", {"aria-label": "PDF"})  # PDF 링크 찾기
        code_link_tag = article.find("a", text="GitHub")  # 코드 URL (GitHub 링크)

        if title_tag and authors_tag:
            title_text = title_tag.text.strip()

            # 저자 정보에서 괄호 속 대학/기관 제거
            authors_raw = ", ".join([a.text.strip() for a in authors_tag.find_all("span")])
            authors_cleaned = re.sub(r"\(.*?\)", "", authors_raw).strip()

            # PDF 링크 정리
            pdf_link = pdf_link_tag["href"] if pdf_link_tag else None
            code_url = code_link_tag["href"] if code_link_tag else None  # 코드 URL 추출

            papers.append({
                "title": title_text,
                "authors": authors_cleaned,
                "pdf_link": pdf_link,
                "code_url": code_url,
                "conference_name": conference_name
            })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers


In [119]:
DB_PATH = "www_conference_2018.db"
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [120]:
page_link = "https://dl.acm.org/doi/proceedings/10.5555/3178876"
conference_name = "WWW 2018"

df_papers = get_www_papers(page_link, conference_name)

# 데이터 확인
df_papers.head()

❌ 페이지를 불러오지 못했습니다.


""


In [100]:
save_to_database(df_papers,conference_name=conference_name, DB_PATH=DB_PATH)

360개의 논문이 WWW 2019에 저장되었습니다.


# KDD 2015

In [2]:
import sqlite3
import requests
from bs4 import BeautifulSoup
import pandas as pd

def create_database(DB_PATH):
    """Create the database and the Conference table."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Conference (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        Title TEXT NOT NULL,
        Author TEXT NOT NULL,
        PDF_Link TEXT,
        Code_URL TEXT,
        Conference_Name TEXT NOT NULL
    )
    ''')

    print("Conference 테이블이 생성되었습니다.")
    conn.commit()
    conn.close()

def save_to_database(df, conference_name, DB_PATH):
    conn = sqlite3.connect(DB_PATH, timeout=10)
    cursor = conn.cursor()

    try:
        for _, row in df.iterrows():  # ✅ iterrows() 사용하여 DataFrame의 각 행을 처리
            # 중복 데이터 확인
            cursor.execute('''
            SELECT 1 FROM Conference WHERE Title = ? AND Author = ? AND Conference_Name = ?
            ''', (row['title'], row['authors'], conference_name))
            result = cursor.fetchone()

            if not result:
                cursor.execute('''
                INSERT INTO Conference (Title, Author, PDF_Link, Code_URL, Conference_Name)
                VALUES (?, ?, ?, ?, ?)
                ''', (row['title'], row['authors'], row['pdf_link'], row['code_url'], conference_name))

        conn.commit()
        print(f"{len(df)}개의 논문이 {conference_name}에 저장되었습니다.")
    except sqlite3.Error as e:
        print(f"Database error: {e}")
    finally:
        conn.close()

In [3]:
url = 'https://dblp.org/db/conf/kdd/kdd2015.html'
DB_PATH = "con_db/KDD_conference_2015.db"
conference_name = 'KDD 2015'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [4]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/KDD_2015_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [20]:
def get_www_papers(html_file_path, conference_name):
    """저장된 HTML 파일에서 KDD 학회의 Accepted Papers 정보를 크롤링 후 DataFrame 반환"""

    # HTML 파일 읽기
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []

    # 논문 리스트가 포함된 섹션 찾기
    for entry in soup.find_all("li", class_="entry inproceedings"):
        # 제목 찾기
        title_tag = entry.find("span", class_="title")
        title_text = title_tag.text.strip() if title_tag else "Unknown"

        # 저자 찾기
        authors_tags = entry.find_all("span", itemprop="author")
        authors_list = [author.find("span", itemprop="name").text.strip() for author in authors_tags]
        authors_cleaned = ", ".join(authors_list)

        # PDF 링크 찾기
        pdf_tag = entry.find("a", href=re.compile(r"doi\.org"))
        pdf_link = pdf_tag["href"] if pdf_tag else None

        # 논문 정보 추가
        papers.append({
            "title": title_text,
            "authors": authors_cleaned,
            "pdf_link": pdf_link,
            "code_url": None,
            "conference_name": conference_name
        })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers

In [21]:
df_papers = get_www_papers('html/KDD_2015_accepted_papers.html',conference_name)

In [22]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Online Controlled Experiments: Lessons from Ru...,Ron Kohavi,https://doi.org/10.1145/2783258.2785464,None,KDD 2015
1,MOOCS: What Have We Learned?,Daphne Koller,https://doi.org/10.1145/2783258.2785465,None,KDD 2015
2,Machine Learning and Causal Inference for Poli...,Susan Athey,https://doi.org/10.1145/2783258.2785466,None,KDD 2015
3,"Data, Knowledge and Discovery: Machine Learnin...",Hugh F. Durrant-Whyte,https://doi.org/10.1145/2783258.2785467,None,KDD 2015
4,Large-Scale Distributed Bayesian Matrix Factor...,"Sungjin Ahn, Anoop Korattikara, Nathan Liu, Su...",https://doi.org/10.1145/2783258.2783373,None,KDD 2015


In [23]:
df_papers = df_papers.drop(index=[0,1,2,3])

In [24]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
4,Large-Scale Distributed Bayesian Matrix Factor...,"Sungjin Ahn, Anoop Korattikara, Nathan Liu, Su...",https://doi.org/10.1145/2783258.2783373,None,KDD 2015
5,TimeMachine: Timeline Generation for Knowledge...,"Tim Althoff, Xin Luna Dong, Kevin Murphy, Safa...",https://doi.org/10.1145/2783258.2783325,None,KDD 2015
6,Estimating Local Intrinsic Dimensionality.,"Laurent Amsaleg, Oussama Chelly, Teddy Furon, ...",https://doi.org/10.1145/2783258.2783405,None,KDD 2015
7,Portraying Collective Spatial Attention in Twi...,"Émilien Antoine, Adam Jatowt, Shoko Wakamiya, ...",https://doi.org/10.1145/2783258.2783418,None,KDD 2015
8,Accelerating Dynamic Time Warping Clustering w...,"Nurjahan Begum, Liudmila Ulanova, Jun Wang, Ea...",https://doi.org/10.1145/2783258.2783286,None,KDD 2015


In [25]:
df_papers.tail()

,title,authors,pdf_link,code_url,conference_name
248,VC-Dimension and Rademacher Averages: From Sta...,"Matteo Riondato, Eli Upfal",https://doi.org/10.1145/2783258.2789984,None,KDD 2015
249,Large Scale Distributed Data Science using Apa...,"James G. Shanahan, Liang Dai",https://doi.org/10.1145/2783258.2789993,None,KDD 2015
250,Medical Mining: KDD 2015 Tutorial.,"Myra Spiliopoulou, Pedro Pereira Rodrigues, Er...",https://doi.org/10.1145/2783258.2789992,None,KDD 2015
251,Big Data Analytics: Optimization and Randomiza...,"Tianbao Yang, Qihang Lin, Rong Jin",https://doi.org/10.1145/2783258.2789989,None,KDD 2015
252,Data Driven Science: SIGKDD Panel.,"Katharina Morik, Hugh F. Durrant-Whyte, Gary C...",https://doi.org/10.1145/2783258.2788703,None,KDD 2015


In [26]:
save_to_database(df_papers, conference_name, DB_PATH)

249개의 논문이 KDD 2015에 저장되었습니다.


# KDD 2014

In [27]:
url = 'https://dblp.org/db/conf/kdd/kdd2014.html'
DB_PATH = "con_db/KDD_conference_2014.db"
conference_name = 'KDD 2014'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [28]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/KDD_2014_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [29]:
df_papers = get_www_papers('html/KDD_2014_accepted_papers.html',conference_name)

In [30]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,The battle for the future of data mining.,Oren Etzioni,https://doi.org/10.1145/2623330.2630816,None,KDD 2014
1,"Data, predictions, and decisions in support of...",Eric Horvitz,https://doi.org/10.1145/2623330.2630815,None,KDD 2014
2,A data driven approach to diagnosing and treat...,Eric E. Schadt,https://doi.org/10.1145/2623330.2630817,None,KDD 2014
3,Bugbears or legitimate threats?: (social) scie...,Sendhil Mullainathan,https://doi.org/10.1145/2623330.2630818,None,KDD 2014
4,Prediction of human emergency behavior and the...,"Xuan Song, Quanshi Zhang, Yoshihide Sekimoto, ...",https://doi.org/10.1145/2623330.2623628,None,KDD 2014


In [31]:
df_papers = df_papers.drop(index=[0,1,2,3])

In [32]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
4,Prediction of human emergency behavior and the...,"Xuan Song, Quanshi Zhang, Yoshihide Sekimoto, ...",https://doi.org/10.1145/2623330.2623628,None,KDD 2014
5,Inferring user demographics and social strateg...,"Yuxiao Dong, Yang Yang, Jie Tang, Yang Yang, N...",https://doi.org/10.1145/2623330.2623703,None,KDD 2014
6,Travel time estimation of a path using sparse ...,"Yilun Wang, Yu Zheng, Yexiang Xue",https://doi.org/10.1145/2623330.2623656,None,KDD 2014
7,Modeling human location data with mixtures of ...,"Moshe Lichman, Padhraic Smyth",https://doi.org/10.1145/2623330.2623681,None,KDD 2014
8,A cost-effective recommender system for taxi d...,"Meng Qu, Hengshu Zhu, Junming Liu, Guannan Liu...",https://doi.org/10.1145/2623330.2623668,None,KDD 2014


In [33]:
save_to_database(df_papers, conference_name, DB_PATH)

216개의 논문이 KDD 2014에 저장되었습니다.
